# Ablation — Transformer + features par articulation

**Objectif :** Mesurer l'impact des vecteurs osseux par articulation sur le Transformer.

Chaque token passe de 3 dims (x,y,z) à 9 dims : xyz + vecteur osseux entrant
(unité) + vecteur osseux sortant (unité). Le reste de l'architecture est identique.

**Protocole expérimental (commun à toutes les expériences) :**
- Dataset : 7 664 images du dataset W7CH (TSL), landmarks MediaPipe (21 points × 2 mains).
- Split stratifié 80/10/10 (seed 42) sur les échantillons ORIGINAUX, **avant**
  toute augmentation → jeux d'entraînement et de validation/test strictement disjoints.
- Augmentation (rotation ±20°, échelle 0.85-1.15, translation ±0.1, bruit 0.02)
  appliquée **uniquement** sur le train, équilibré à 200 échantillons/classe.
- Entraînement : Adam (lr 1e-3, wd 1e-4), CrossEntropy + label smoothing 0.1,
  ReduceLROnPlateau, early stopping sur val_acc, seeds fixes.
- 56 classes effectives sur 57 (les images `metro` ne sont pas détectées par
  MediaPipe), tête de classification à 57 sorties (comme le modèle actuel).


In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
while not (ROOT / "common").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for p in [str(ROOT), str(ROOT / "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from common.paths import RESULTS_DIR, MODELS_DIR, PLOTS_DIR, REPO_ROOT
from common.data import load_dataset, normalized_inputs
from common.models import (
    count_params, set_seed, BaselineMLP, LandmarkTransformer, HandGCN, HandGAT,
)
from common.train import (
    Trainer, predict, full_metrics, save_metrics,
    plot_training_curves, plot_confusion, run_torch_benchmark,
)
from common import export as ex
from common.robustness import robustness_summary

set_seed(42)

def load_class_names():
    with open(REPO_ROOT / "data" / "processed" / "class_names.txt", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

class_names = load_class_names()
print("torch", torch.__version__)


torch 2.13.0+cpu


## 1. Chargement des données

Le dataset préparé (split + augmentation, cf. README) est chargé. Le modèle
reçoit 42 tokens × 9 dims (xyz + os entrant + os sortant).


In [2]:
dataset = load_dataset()
from common.data import joint_features
jt_train = joint_features(dataset["train"]["img_landmarks"])
jt_val   = joint_features(dataset["val"]["img_landmarks"])
jt_test  = joint_features(dataset["test"]["img_landmarks"])
X_train, y_train = torch.from_numpy(jt_train).float(), torch.from_numpy(dataset["train"]["labels"])
X_val, y_val     = torch.from_numpy(jt_val).float(),   torch.from_numpy(dataset["val"]["labels"])
X_test, y_test   = torch.from_numpy(jt_test).float(),  torch.from_numpy(dataset["test"]["labels"])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)
print("train:", tuple(X_train.shape), "| val:", tuple(X_val.shape), "| test:", tuple(X_test.shape))


train: (13282, 2, 21, 9) | val: (767, 2, 21, 9) | test: (767, 2, 21, 9)


## 2. Architecture

Transformer identique au notebook 02, avec tokens à 9 dimensions.

Nombre de paramètres : mesuré ci-dessous.


In [3]:
model = LandmarkTransformer(num_classes=57, d_model=64, nhead=4, num_layers=2, dim_ff=128, token_dim=9)
print(f"params: {count_params(model):,}")


params: 79,865


## 3. Entraînement

Mêmes hyperparamètres que le protocole commun (Adam lr 1e-3, wd 1e-4,
label smoothing 0.1, early stopping patience 20, max 60 epochs).


In [4]:
trainer = Trainer(model, patience=20)
history = trainer.fit(train_loader, val_loader, max_epochs=60, early_stop=True)
trainer.restore_best()

plot_training_curves(history, PLOTS_DIR / "training_curves" / "transformer_geo_curves.png",
                     title="Ablation — Transformer + features par articulation")
print("best val acc:", trainer.best_val_acc)


  ep 1/60 train 4.0137/0.0418 val 3.8579/0.0678 (8.6s) best 0.0678


  ep 2/60 train 3.4825/0.1223 val 2.9828/0.2412 (8.1s) best 0.2412


  ep 3/60 train 2.9384/0.2443 val 2.4963/0.3937 (8.4s) best 0.3937


  ep 4/60 train 2.6137/0.3484 val 2.1429/0.5033 (8.1s) best 0.5033


  ep 5/60 train 2.3647/0.4271 val 1.9425/0.5880 (8.3s) best 0.5880


...

  ep 57/60 train 1.1527/0.8803 val 1.0136/0.9309 (8.8s) best 0.9348


  ep 58/60 train 1.1461/0.8804 val 1.0220/0.9361 (8.8s) best 0.9361


  ep 59/60 train 1.1501/0.8773 val 1.0050/0.9361 (8.5s) best 0.9361


  ep 60/60 train 1.1443/0.8830 val 1.0011/0.9478 (8.9s) best 0.9478


best val acc: 0.9478487614080835


## 4. Évaluation sur le jeu de test

Métriques complètes : accuracy, precision/recall/F1 (macro et pondérés),
matrice de confusion, performance par classe.


In [5]:
y_pred, y_true = predict(model, test_loader)
metrics = full_metrics(y_true, y_pred, num_classes=57)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"macro F1: {metrics['macro_f1']:.4f} | weighted F1: {metrics['f1']:.4f}")
print(f"macro precision: {metrics['macro_precision']:.4f} | macro recall: {metrics['macro_recall']:.4f}")

cm = np.array(metrics["confusion_matrix"])
plot_confusion(cm, class_names, PLOTS_DIR / "confusion_matrices" / "transformer_geo_cm.png",
               title="Ablation — Transformer + features par articulation — confusion matrix (test)")
np.save(RESULTS_DIR / "confusion_transformer_geo.npy", cm)

import pandas as pd
per = pd.DataFrame({
    "class": class_names,
    "precision": metrics["per_class_precision"],
    "recall": metrics["per_class_recall"],
    "f1": metrics["per_class_f1"],
    "support": metrics["per_class_support"],
})
per.to_csv(RESULTS_DIR / "per_class_transformer_geo.csv", index=False)
print(per.sort_values("f1").head(8))


test accuracy: 0.9309
macro F1: 0.9320 | weighted F1: 0.9303
macro precision: 0.9358 | macro recall: 0.9359


     class  precision    recall        f1  support
54   metro   0.000000  0.000000  0.000000        0
22  telvza   0.750000  0.500000  0.600000        6
17  siye7a   0.777778  0.777778  0.777778        9
38     jad   0.869565  0.740741  0.800000       27
41     o5t   0.913043  0.724138  0.807692       29
39   jadda   0.714286  0.937500  0.810811       16
4   chabeb   0.777778  0.875000  0.823529        8
15   radio   0.857143  0.818182  0.837209       22


## 5. Robustesse

Le modèle est évalué sous perturbations des landmarks appliquées **avant**
normalisation (dans l'espace image) : bruit gaussien, rotation, translation,
changement d'échelle — 5 essais par perturbation, moyenne ± écart-type.


In [6]:
def forward_fn(model, raw, batch_size=512):
    # Perturbed raw landmarks -> per-joint features (x,y,z + bone vectors)
    model.eval()
    from common.data import joint_features
    jf = joint_features(raw)
    outs = []
    with torch.no_grad():
        for i in range(0, len(jf), batch_size):
            x = torch.from_numpy(jf[i:i + batch_size]).float()
            outs.append(model(x).argmax(1).numpy())
    return np.concatenate(outs)
rob = robustness_summary(model, dataset["test"]["img_landmarks"],
                         dataset["test"]["labels"], base_acc=metrics["accuracy"],
                         forward_fn=forward_fn)


  noise_0.01: acc=0.9241 (drop=+0.0068)


  noise_0.02: acc=0.8780 (drop=+0.0529)


  noise_0.05: acc=0.6248 (drop=+0.3061)


  rotation_5: acc=0.9374 (drop=-0.0065)


  rotation_10: acc=0.9322 (drop=-0.0013)


  rotation_20: acc=0.8866 (drop=+0.0443)


  translation_0.05: acc=0.9317 (drop=-0.0008)


  translation_0.10: acc=0.9173 (drop=+0.0136)


  scale_0.90: acc=0.9296 (drop=+0.0013)


  scale_1.10: acc=0.9387 (drop=-0.0078)


## 6. Mesures mobiles

- Nombre de paramètres et taille du checkpoint PyTorch.
- Export ONNX (FP32), vérification, latence ONNX Runtime CPU (batch=1).
- Versions FP16 et INT8 (quantification dynamique) : taille, latence,
  dégradation d'accuracy sur le jeu de test.


In [7]:
sample = X_test[:1].float()
lat_torch = run_torch_benchmark(model, sample)
print(f"torch latency: {lat_torch:.3f} ms/sample")

onnx_path = ex.export_onnx(model, sample, "transformer_geo")
fp16 = ex.quantize_fp16(onnx_path, "transformer_geo")
int8 = ex.quantize_int8(onnx_path, "transformer_geo")

quantized = {}
for tag, p in [("fp32", onnx_path), ("fp16", fp16), ("int8", int8)]:
    preds = ex.ort_predict(p, X_test.numpy())
    quantized[tag] = {
        "size_mb": ex.file_size_mb(p),
        "latency_ms": ex.ort_latency(p, sample.numpy()),
        "test_acc": float((preds == y_test.numpy()).mean()),
    }
    print(f"  {tag}: size={quantized[tag]['size_mb']:.2f} MB "
          f"latency={quantized[tag]['latency_ms']:.3f} ms "
          f"acc={quantized[tag]['test_acc']:.4f}")

extra = {
    "robustness": rob,
    "epochs": trainer.epochs_run,
    "params": count_params(model),
    "best_val_acc": trainer.best_val_acc,
    "torch_latency_ms": lat_torch,
    "quantized": quantized,
}
save_metrics("transformer_geo", metrics, extra=extra)

torch_path = ex.save_torch(model, "transformer_geo")
print("saved:", torch_path, f"({ex.file_size_mb(torch_path):.2f} MB)")


torch latency: 0.543 ms/sample

  fp32: size=0.34 MB latency=0.191 ms acc=0.9309
  fp16: size=0.18 MB latency=0.196 ms acc=0.9309


  int8: size=0.15 MB latency=0.275 ms acc=0.9335
metrics saved -> D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\results\metrics_transformer_geo.json
saved: D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\models\transformer_geo.pth (0.32 MB)


## 7. Conclusions

Comparaison transformer vs transformer+geo : les vecteurs osseux unitaires sont
invariants à l'échelle — ils peuvent aider le Transformer qui reçoit des coordonnées absolues.
